# EXP-018 — SVM Feature Selection trên WESAD

**Người thực hiện:** Nam  
**Nhóm:** 04 — Cải tiến lựa chọn đặc trưng  
**Trạng thái:** Chưa chạy trên dữ liệu thật (chỉ ghi kết quả sau khi chạy)  
**Mục tiêu:** So sánh SVM mặc định RBF trên **68 đặc trưng** và SVM trên **30 đặc trưng** chọn bằng SelectKBest (ANOVA F), cùng outer LOSO trên 15 người.

**Giả thuyết:** Loại bỏ đặc trưng dư thừa/ít liên quan có thể cải thiện Macro-F1; đây là giả thuyết cần kiểm nghiệm, không phải kết luận trước.


## 2. Thiết kế thí nghiệm và kiểm soát rò rỉ

- Input: G2 physiological68, 15 subject; binary: Stress=1 và không Stress=0; multiclass: nhãn 1/2/3.
- Outer CV: `LeaveOneGroupOut(subject_id)`; train 14 người, test 1 người.
- Hai biến thể: `svm_68` và `svm_top30` (`C=1`, `kernel=rbf`, `gamma=scale`, **không** class balancing hay tuning).
- Trong **mỗi outer fold**: fit MedianImputer → StandardScaler → SelectKBest (chỉ top30) → SVC **chỉ trên train**. Không chọn feature trước khi chia LOSO.
- So sánh công bằng: hai biến thể đánh giá đúng cùng tập test. EXP-004 là baseline bên ngoài, chỉ đối chiếu nếu schema và protocol giống nhau.
- Metrics: Macro-F1, Balanced Accuracy; xuất theo người, trung bình ± độ lệch chuẩn và Confusion Matrix.


## 3. Khởi tạo và đọc cấu hình

In [5]:
from pathlib import Path
import sys, json, subprocess
import pandas as pd
import yaml
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/run_experiment.py").is_file() and (p / "src/wesad/data.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Không tìm thấy gốc wesad-stress-detection; hãy mở notebook trong đúng dự án.")
EXP = ROOT / "experiments/04_additional/EXP-018_SVM"
CONFIG_PATH = EXP / "config.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
print("Dữ liệu:", CONFIG["features"])
print("Top-k:", CONFIG["selected_features"])


Dữ liệu: data/features/g2_physiological68_group15_20261107/all_subjects.csv
Top-k: 30


## 4. Kiểm tra dữ liệu và mô hình

In [6]:
sys.path.insert(0, str(ROOT / "src"))
from wesad.data import load_feature_table
from sklearn.svm import SVC
from sklearn.feature_selection import SelectKBest, f_classif

feature_file = ROOT / CONFIG["features"]
schema_file = ROOT / CONFIG["schema"]
print("Data exists:", feature_file.is_file(), "Schema exists:", schema_file.is_file())
if feature_file.exists() and schema_file.exists():
    from hashlib import sha256
    actual = sha256(feature_file.read_bytes()).hexdigest()
    assert actual == CONFIG["feature_sha256"], "Checksum WESAD không khớp cấu hình!"
    frame, feature_cols = load_feature_table(feature_file, schema_file)
    assert len(feature_cols) == 68 and frame.subject_id.nunique() == 15
    print("Số cửa sổ:", len(frame), "| Số người:", frame.subject_id.nunique(), "| Số features:", len(feature_cols))
    display(frame.label_original.value_counts().rename_axis("nhãn").reset_index(name="số cửa sổ"))
else:
    print("Chưa có dataset trên máy. Xem README_EXP018.md.")



Data exists: True Schema exists: True
Số cửa sổ: 6189 | Số người: 15 | Số features: 68


,nhãn,số cửa sổ
0,1,3370
1,2,1848
2,3,971


## 5. R01 — Kiểm tra nhanh trên người S2

Chỉ bật `RUN_QUICK=True` để chạy hai bài toán × hai biến thể trên S2. Kết quả thử nằm trong `run_outputs/quick_S2/` và **không được coi là kết quả đủ 15 người**.


In [7]:
RUN_QUICK = True
if RUN_QUICK:
    command=[sys.executable, str(ROOT / "scripts/run_exp018.py"), "--config",str(CONFIG_PATH), "--quick-subject","S2"]
    result=subprocess.run(command,cwd=ROOT,capture_output=True,text=True,encoding="utf-8",errors="replace")
    print(result.stdout)
    if result.returncode:
        print(result.stderr)
        raise RuntimeError(f"R01 thất bại: returncode={result.returncode}")
else:
    print("RUN_QUICK=False. Không chạy thử.")



OpenBLAS error: Memory allocation still failed after 10 retries, giving up.
OpenBLAS error: Memory allocation still failed after 10 retries, giving up.
OpenBLAS error: Memory allocat


RuntimeError: R01 thất bại: returncode=1

## 6. R02 — Thí nghiệm chính thức đủ 15 người

Đặt `RUN_MAIN=True` để chạy **60 lượt đánh giá**: 15 người × 2 bài toán × 2 biến thể. Cần đúng dữ liệu G2 SHA-256 đã chốt. Kết quả sẽ tự lưu **ngay trong thư mục EXP-018** (`results.csv`, `summary.csv`, `predictions.csv`, `selected_features_by_fold.csv`, `feature_selection_frequency.csv`, `comparison_delta.csv`, `figures/*.png`).


In [ ]:
RUN_MAIN = True
if RUN_MAIN:
    command=[sys.executable,str(ROOT / "scripts/run_exp018.py"),"--config",str(CONFIG_PATH)]
    result=subprocess.run(command,cwd=ROOT,capture_output=True,text=True,encoding="utf-8",errors="replace")
    print(result.stdout[-10000:])
    if result.returncode:
        print("STDERR:\n",result.stderr)
        raise RuntimeError(f"R02 thất bại: returncode={result.returncode}")
else:
    print("RUN_MAIN=False. Không chạy chính thức.")


## 7. Đọc kết quả và xác minh trước khi kết luận

In [ ]:
summary_path = EXP / "summary.csv"
provenance_path = EXP / "result_provenance.json"
if summary_path.is_file() and provenance_path.is_file():
    prov=json.loads(provenance_path.read_text(encoding="utf-8"))
    summary=pd.read_csv(summary_path)
    if prov.get("status") != "complete":
        print("Lưu ý: chỉ xem trước; provenance chưa xác nhận main complete")
    assert set(summary["variant"]) == {"svm_68", "svm_top30"}
    assert (summary.n_subjects == 15).all() and len(summary)==4
    display(summary.round(4))
else:
    print("Chưa có kết quả main hợp lệ. Chạy R02 trước.")


## 8. Biểu đồ, chọn đặc trưng và so sánh

In [ ]:
from IPython.display import Image
if (EXP / "feature_selection_frequency.csv").is_file():
    freq=pd.read_csv(EXP / "feature_selection_frequency.csv")
    display(freq.sort_values("n_folds_selected",ascending=False).head(12))
if (EXP / "comparison_delta.csv").is_file():
    display(pd.read_csv(EXP / "comparison_delta.csv").round(4))
for name in ("comparison_feature_selection.png", "macro_f1_by_subject_binary.png", "macro_f1_by_subject_multiclass.png"):
    f=EXP / "figures" / name
    if f.exists(): display(Image(filename=str(f)))


## 9. Đối chiếu với EXP-004 — SVM Baseline

SVM 68 trong EXP-018 là **đối chứng nội bộ** tốt nhất vì dùng chính xác cùng train/test, dữ liệu và code như biến thể top30. Có thể đọc thêm `EXP-004_SVM/summary.csv` để kiểm tra kết quả baseline độc lập, nhưng phải đối chiếu checksum, tiền xử lý, số người và cách tính metric trước khi kết luận.


In [ ]:
exp004 = ROOT / "experiments/01_baselines/EXP-004_SVM/summary.csv"
if exp004.exists():
    print("EXP-004 có summary:", exp004)
    display(pd.read_csv(exp004).head(8))
else:
    print("Chưa có summary EXP-004 cùng máy; dùng đối chứng nội bộ svm_68.")


## 10. Hạn chế và độ tin cậy

- 15 người, cửa sổ tín hiệu chồng lấn; không xem cửa sổ là mẫu độc lập khi suy luận thống kê.
- Số lượng đặc trưng 30 được chọn trước, chưa tối ưu top-k. ANOVA F xét quan hệ đơn biến, không chứng minh quan hệ nhân quả.
- Cần báo cáo kết quả từng người và độ lệch chuẩn. Không tự suy ra kiểm định ý nghĩa thống kê từ chênh lệch trung bình.
- Không dùng EXP-013 SVM cân bằng lớp làm đối chứng chính vì khác cơ chế học; baseline đúng là SVM không cân bằng.


## 11. Nhận xét và kết luận

**Chỉ điền sau khi R02 chạy và xác minh đầy đủ:**

- Binary: `svm_68` Macro-F1 = ...; `svm_top30` = ...; Δ = ...; Balanced Accuracy Δ = ... .
- Multiclass: `svm_68` Macro-F1 = ...; `svm_top30` = ...; Δ = ...; Balanced Accuracy Δ = ... .
- Nhận xét độ ổn định: so sánh độ lệch chuẩn và biểu đồ theo 15 người.
- Phân tích top features theo `feature_selection_frequency.csv` (tần suất qua 15 fold).
- **Kết luận phải dựa trên CSV thực tế**, không khẳng định cải thiện nếu chưa thấy kết quả.
